# Stage 4 - Embedding diagnostics

Runs the Stage 4 analysis interactively on the Stage 3 checkpoint (test split only). See `results/stage4_notes.md` for the write-up; this notebook lets you re-run pieces and poke at intermediate arrays.

In [ ]:
import sys, os, json, torch
sys.path.insert(0, os.path.abspath('../src'))
from match import load_model
from evaluate import collect, embed_paths
import diagnostics as diag

In [ ]:
device = torch.device('cpu')
model, cfg = load_model('../models/best_model.pt', device)
sp = json.load(open('../models/splits.json'))
graph_dir = cfg['data']['graph_dir']
gallery, probes = collect(graph_dir, sp['test'], packed='../data/packed/test.pt')
len(gallery), len(probes)

In [ ]:
gf_ids = list(gallery)
fidx = {f: i for i, f in enumerate(gf_ids)}
ge = embed_paths(model, [gallery[f] for f in gf_ids], device)
gf = torch.tensor([fidx[f] for f in gf_ids])
pe = embed_paths(model, [p for p, _, _ in probes], device)
pf = torch.tensor([fidx[f] for _, f, _ in probes])
tags = [t for _, _, t in probes]

## Gallery inter-identity similarity
How confusable are different fingers' embeddings? A heavy right tail here caps Rank-1 regardless of probe quality.

In [ ]:
diag.gallery_intra_similarity(ge, '../results/stage4')

## t-SNE of embeddings
Star = gallery (Real print), dot = probe (altered). Color = finger identity.

In [ ]:
diag.tsne_plot(ge, gf, gf_ids, pe, pf, tags, '../results/stage4')

## Genuine score by alteration severity

In [ ]:
diag.score_hist_by_tag(pe, pf, ge, gf, tags, '../results/stage4')

## Rank-1 decision margin: near-misses vs blowouts

In [ ]:
diag.rank_gap_analysis(pe, pf, ge, gf, '../results/stage4')